# CAD-Gym — Gamified RL Training (Colab)

Train a language model to design mechanical brackets and watch it improve in real-time.

**What you'll see:**
- Live 3D rotating view of each generated bracket
- Score gauges (structural / cost / geometry)
- Learning curves updating every 10 episodes
- Achievement unlocks as the model improves
- Champion bracket visualization at the end

**Runtime:** GPU is optional. TinyLlama runs fine on Colab CPU (slower) or T4 GPU.

---

## 1. Install

In [ ]:
!pip install -q cadquery trimesh gymnasium pydantic
!pip install -q plotly transformers torch accelerate tqdm
!pip install -q git+https://github.com/prodata-ai/ProdataGym.git

import plotly.io as pio
pio.renderers.default = "colab"

print("Installation complete")

## 2. Load environment + model

In [ ]:
import gymnasium as gym
import prodata.cad_gym
from prodata.cad_gym.viz import TrainingVisualizer

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.optim import AdamW
from tqdm.notebook import tqdm

env = gym.make("prodata/BracketDesign-v0")
print(f"Environment: {len(env.unwrapped.task_ids())} tasks loaded")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Swap for Qwen/Qwen2.5-Coder-7B-Instruct for much better results

print(f"Loading {MODEL_NAME}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)

optimizer = AdamW(model.parameters(), lr=2e-5)
print(f"Model loaded ({sum(p.numel() for p in model.parameters())//1e6:.0f}M params)")

## 3. Zero-shot baseline

Run 5 tasks before any training to establish a baseline score.

In [ ]:
def build_prompt(obs: dict) -> str:
    return (
        "<|system|>\n"
        "You are a mechanical engineer. Write Python code using CadQuery to design "
        "a bracket. Output ONLY valid Python code. No explanation.\n"
        "<|user|>\n"
        f"{obs['task_description']}\n\n"
        f"Load: {obs['load_kg'][0]:.0f} kg | "
        f"Extension: {obs['extension_mm'][0]:.0f} mm | "
        f"Budget: ${obs['max_cost_usd'][0]:.0f}\n\n"
        "Rules:\n"
        "- Start with: import cadquery as cq\n"
        "- Define a variable named `result` of type cadquery.Workplane\n"
        "<|assistant|>\n"
        "```python\n"
        "import cadquery as cq\n"
        "result = "
    )


@torch.no_grad()
def generate_code(obs: dict, temperature: float = 0.8) -> str:
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    if "```" in raw:
        raw = raw.split("```")[0]
    return f"import cadquery as cq\nresult = {raw.strip()}"


def show_bracket(stl_path: str, title: str, color: str = "steelblue"):
    """Wireframe + flat surface render. No specular highlights or reflections."""
    import trimesh
    import plotly.graph_objects as go

    mesh = trimesh.load(stl_path).subdivide()
    v, f = mesh.vertices, mesh.faces

    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        color=color, opacity=0.25, flatshading=True,
        lighting=dict(ambient=1.0, diffuse=0.1, specular=0.0),
        showscale=False,
    ))
    edges = mesh.edges_unique
    xe, ye, ze = [], [], []
    for e in edges:
        xe += [v[e[0],0], v[e[1],0], None]
        ye += [v[e[0],1], v[e[1],1], None]
        ze += [v[e[0],2], v[e[1],2], None]
    fig.add_trace(go.Scatter3d(
        x=xe, y=ye, z=ze, mode="lines",
        line=dict(color="steelblue", width=1), hoverinfo="skip",
    ))
    fig.update_layout(
        title=title, showlegend=False,
        scene=dict(aspectmode="data"),
        width=640, height=460,
        margin=dict(l=0, r=0, t=50, b=0),
    )
    fig.show()


# ── Baseline run ──────────────────────────────────────────────────────────────
print("Zero-shot baseline (5 tasks)\n" + "-" * 40)
baseline_tasks = env.unwrapped.task_ids()[:5]
baseline_results = []

for tid in baseline_tasks:
    obs, _ = env.reset(options={"task_id": tid})
    code = generate_code(obs)
    _, reward, _, _, info = env.step(code)
    baseline_results.append(info["success"])
    dim = info["dimension_scores"]
    status = "PASS" if info["success"] else "FAIL"
    print(f"  {tid}  {status}  reward={reward:.2f}  "
          f"S={dim.get('structural',0):.2f} C={dim.get('cost',0):.2f} G={dim.get('geometry',0):.2f}")

print(f"\nBaseline pass rate: {sum(baseline_results)}/{len(baseline_results)} = "
      f"{sum(baseline_results)/len(baseline_results):.0%}")

## 4. RL Training with live dashboard

Dashboard updates every 10 episodes. Rotate 3D models with your mouse.

In [ ]:
N_EPISODES = 200    # Increase to 500+ for meaningful improvement with TinyLlama
VIZ_EVERY  = 10

viz = TrainingVisualizer(
    target_pass_rate=0.50,
    rolling_window=30,
)

model.train()
print(f"Training {N_EPISODES} episodes | dashboard every {VIZ_EVERY}\n")

for episode in tqdm(range(1, N_EPISODES + 1), desc="Training"):

    obs, info = env.reset()
    prompt    = build_prompt(obs)
    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        gen_out = model.generate(
            **inputs, max_new_tokens=250, temperature=0.8,
            do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = gen_out[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    if "```" in raw:
        raw = raw.split("```")[0]
    code = f"import cadquery as cq\nresult = {raw.strip()}"

    obs, reward, terminated, truncated, info = env.step(code)
    newly_unlocked = viz.update(episode, obs, reward, info, code)

    # REINFORCE with reward shifting (crashes -> 0 gradient, passes -> positive)
    shifted_reward = reward - (-1.0)   # maps [-1, 1] -> [0, 2]
    optimizer.zero_grad()
    with torch.enable_grad():
        full_ids = gen_out[0].unsqueeze(0)
        labels   = full_ids.clone()
        labels[0, :inputs["input_ids"].shape[-1]] = -100
        out  = model(input_ids=full_ids, labels=labels)
        loss = out.loss * shifted_reward
        if shifted_reward > 0.01:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    if episode % VIZ_EVERY == 0 or newly_unlocked:
        viz.render(episode)

print("Training complete")

## 5. Final summary + champion design

In [ ]:
viz.final_summary()

## 6. Post-training evaluation on all 50 tasks

In [ ]:
model.eval()
task_ids = env.unwrapped.task_ids()
results  = {}

print("Evaluating on all 50 tasks...")
for tid in tqdm(task_ids, desc="Eval"):
    obs, _ = env.reset(options={"task_id": tid})
    code   = generate_code(obs, temperature=0.1)
    _, reward, _, _, info = env.step(code)
    results[tid] = {
        "passed": info["success"],
        "reward": round(reward, 4),
        "dimension_scores": info["dimension_scores"],
    }

import json
from pathlib import Path
tasks_path = Path(prodata.cad_gym.__file__).parent / "tasks" / "brackets_basic.json"
with open(tasks_path) as f:
    tasks_meta = {t["task_id"]: t for t in json.load(f)}

print(f"\n{'='*60}")
print(f"Post-RL Evaluation — {MODEL_NAME}")
print(f"{'='*60}")
for level in ("easy", "medium", "hard"):
    ids    = [tid for tid, t in tasks_meta.items() if t["difficulty"] == level]
    passed = sum(results[tid]["passed"] for tid in ids if tid in results)
    avg_r  = np.mean([results[tid]["reward"] for tid in ids if tid in results])
    print(f"  {level.upper():8s}: {passed}/{len(ids)} passed  avg reward {avg_r:.3f}")

total_pass = sum(r["passed"] for r in results.values())
total_avg  = np.mean([r["reward"] for r in results.values()])
print(f"{'='*60}")
print(f"  TOTAL: {total_pass}/{len(results)} = {total_pass/len(results):.1%}  avg reward {total_avg:.3f}")
print(f"  Improvement: {total_pass/50 - sum(baseline_results)/5:+.1%} vs zero-shot")

## 7. Before vs After — side-by-side design comparison

In [ ]:
import trimesh
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def wireframe_traces(stl_path: str, surface_color: str, wire_color: str):
    """Returns (surface_trace, wireframe_trace) for a mesh file."""
    mesh = trimesh.load(stl_path).subdivide()
    v, f = mesh.vertices, mesh.faces

    surface = go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        color=surface_color, opacity=0.25, flatshading=True,
        lighting=dict(ambient=1.0, diffuse=0.1, specular=0.0),
        showscale=False,
    )
    edges = mesh.edges_unique
    xe, ye, ze = [], [], []
    for e in edges:
        xe += [v[e[0],0], v[e[1],0], None]
        ye += [v[e[0],1], v[e[1],1], None]
        ze += [v[e[0],2], v[e[1],2], None]
    wireframe = go.Scatter3d(
        x=xe, y=ye, z=ze, mode="lines",
        line=dict(color=wire_color, width=1), hoverinfo="skip",
    )
    return surface, wireframe


best_task_id = viz.best["task_id"]
best_mesh    = viz.best["mesh_file"]

if best_task_id and best_mesh:
    # Generate a current zero-shot attempt on the same task
    obs, _ = env.reset(options={"task_id": best_task_id})
    zs_code = generate_code(obs, temperature=0.01)
    _, zs_reward, _, _, zs_info = env.step(zs_code)
    zs_mesh = zs_info.get("mesh_file")

    if zs_mesh:
        # Two subplots, each gets surface + wireframe
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scene"}, {"type": "scene"}]],
            subplot_titles=(
                f"Zero-shot  reward={zs_reward:.2f}",
                f"Best trained  reward={viz.best['reward']:.2f}",
            ),
        )

        s1, w1 = wireframe_traces(zs_mesh,   "salmon",     "firebrick")
        s2, w2 = wireframe_traces(best_mesh, "lightblue",  "steelblue")

        fig.add_trace(s1, row=1, col=1)
        fig.add_trace(w1, row=1, col=1)
        fig.add_trace(s2, row=1, col=2)
        fig.add_trace(w2, row=1, col=2)

        fig.update_layout(
            title_text=f"Task {best_task_id} — Before vs After",
            height=500, width=960, showlegend=False,
            scene=dict(aspectmode="data"),
            scene2=dict(aspectmode="data"),
        )
        fig.show()
    else:
        print("Zero-shot attempt crashed — no mesh to compare.")
        if best_mesh:
            show_bracket(best_mesh, f"Best design — {best_task_id}  reward={viz.best['reward']:.2f}")
else:
    print("No passing design found during training. Try more episodes.")

## Next steps

- **More episodes** — TinyLlama needs 300-500 episodes to show meaningful improvement.
- **Bigger model** — Swap `TinyLlama` for `Qwen/Qwen2.5-Coder-7B-Instruct` for much better results.
- **Pro verifier** — Run with `verifier_mode="pro"` to avoid reward hacking. Sign up at [prodata.ai](https://prodata.ai).
- **Leaderboard** — Open a PR to [github.com/prodata-ai/ProdataGym](https://github.com/prodata-ai/ProdataGym) with your results.